# 3. Model Training and Evaluation
This notebook evaluates five distinct machine learning models (Logistic Regression, KNN, Decision Tree, Random Forest, SVM) using 5-fold stratified cross-validation. It then selects the best-performing model based on Recall and F1-score, and saves the full deployment pipeline.

In [ ]:
import pandas as pd
import numpy as np
import json
import joblib
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, confusion_matrix

# Algorithms
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# Load dataset
df = pd.read_csv('student_performance_expanded.csv')
df_orig = df[df['data_origin'] == 'Original'].copy()
df_synth = df[df['data_origin'] == 'Synthetic'].copy()

train_orig, test_orig = train_test_split(df_orig, test_size=0.25, random_state=42, stratify=df_orig['at_risk'])
df_train = pd.concat([train_orig, df_synth], ignore_index=True)
df_test = test_orig.copy()

FEATURES = [
    'age', 'gender', 'department', 'semester', 'study_hours_per_week',
    'attendance_percentage', 'assignment_average', 'midterm_score',
    'previous_gpa', 'internet_access', 'extra_academic_support',
    'part_time_job', 'extracurricular_hours_per_week', 'absences'
]
NUM_FEATURES = [
    'age', 'semester', 'study_hours_per_week', 'attendance_percentage',
    'assignment_average', 'midterm_score', 'previous_gpa',
    'extracurricular_hours_per_week', 'absences'
]
CAT_FEATURES = ['gender', 'department', 'internet_access', 'extra_academic_support', 'part_time_job']

X_train = df_train[FEATURES]
y_train = df_train['at_risk']
X_test = df_test[FEATURES]
y_test = df_test['at_risk']

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), NUM_FEATURES),
    ('cat', OneHotEncoder(handle_unknown='ignore'), CAT_FEATURES)
])

### Evaluate 5 Algorithms using Stratified 5-Fold Cross Validation

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1500, class_weight='balanced', random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5),
    'Decision Tree': DecisionTreeClassifier(max_depth=6, class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=150, max_depth=7, class_weight='balanced', random_state=42),
    'Support Vector Machine': SVC(probability=True, class_weight='balanced', random_state=42)
}

results = {}
for name, clf in models.items():
    pipeline = Pipeline([('preprocessor', preprocessor), ('model', clf)])
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    f1s, recs, precs = [], [], []
    for train_idx, val_idx in skf.split(X_train, y_train):
        X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
        X_va, y_va = X_train.iloc[val_idx], y_train.iloc[val_idx]
        
        pipeline.fit(X_tr, y_tr)
        val_preds = pipeline.predict(X_va)
        f1s.append(f1_score(y_va, val_preds))
        recs.append(recall_score(y_va, val_preds))
        precs.append(precision_score(y_va, val_preds))
        
    results[name] = {
        'CV Mean F1': np.mean(f1s),
        'CV Mean Recall': np.mean(recs),
        'CV Mean Precision': np.mean(precs)
    }

print(pd.DataFrame(results).T)

### Train on full training set and evaluate on untouched test set

In [ ]:
metrics_list = []
fitted_pipelines = {}

for name, clf in models.items():
    pipeline = Pipeline([('preprocessor', preprocessor), ('model', clf)])
    pipeline.fit(X_train, y_train)
    fitted_pipelines[name] = pipeline
    
    preds = pipeline.predict(X_test)
    probs = pipeline.predict_proba(X_test)[:, 1]
    
    metrics_list.append({
        'model': name,
        'accuracy': round(accuracy_score(y_test, preds), 4),
        'precision': round(precision_score(y_test, preds), 4),
        'recall': round(recall_score(y_test, preds), 4),
        'f1_score': round(f1_score(y_test, preds), 4),
        'roc_auc': round(roc_auc_score(y_test, probs), 4),
        'pr_auc': round(average_precision_score(y_test, probs), 4)
    })

df_metrics = pd.DataFrame(metrics_list)
print(df_metrics)